# Proyecto final — Mercado de vehículos ligeros en México

**Equipo:** José Morfín Suárez, ______, ______  
**Materia:** Inteligencia de Negocio y Soluciones de Ciencia de Datos (Prof. Edgar Avalos Gauna)  
**Fuentes:** (1) base de datos de la app de Streamlit de la clase *(obligatoria, pendiente de recibir)*; (2) Reporte Mercado Interno Automotor Ligeros, AMDA, agosto 2026, con cifras de INEGI *(complemento)*.

Los datos del reporte están en la carpeta `datos/` del repositorio
[github.com/JoseMorfin/bi-proyecto-final-amda](https://github.com/JoseMorfin/bi-proyecto-final-amda).
Este cuaderno los lee directo de ahí; no hay que subir nada a Colab.

## Cómo califica Edgar (40% del curso + 10% dashboard)

| Sección de este cuaderno | Criterio | Peso |
|---|---|---|
| 1 | Propuesta de negocio | 5% |
| 2 | Análisis de datos y metadatos | 5% |
| 3 | EDA (análisis exploratorio) | 10% |
| 4 | Implementación de técnica de ciencia de datos | 10% |
| 5 | Data storytelling y visualización | 5% |
| 6 | Conclusiones y aportaciones | 5% |
| 7 | Dashboard en Streamlit (aparte) | 10% |

Presentación: **15 minutos ± 2**. Fuera de ese rango, −10% del proyecto.

## 0. Importar librerías y cargar los datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


Se leen directo del repositorio (liga *raw* de GitHub), igual que Edgar hizo con el mapa de Washington.
Empieza con ventas mensuales; los demás archivos se cargan donde se usan cambiando el nombre.

In [ ]:
base = "https://raw.githubusercontent.com/JoseMorfin/bi-proyecto-final-amda/main/datos/"
data = pd.read_csv(base + "ventas_mensuales_2008_2026.csv")
data.head()


## 1. Propuesta de negocio (5%)
Una hipótesis clara, de una o dos líneas, que represente una oportunidad para alguien real (una agencia, una armadora, una financiera, un gobierno estatal).
Escríbanla con sus palabras. Tres ideas para escoger o combinar:

- **Inventario:** los compactos van a seguir cayendo y los SUV e híbridos subiendo; una agencia que reasigne su inventario a tiempo vende más y descuenta menos.
- **Pronóstico:** con la serie 2008-2026 se puede pronosticar el cierre de 2026 y compararlo con el pronóstico oficial de AMDA (viene en `pronostico_amda_2026.csv`), para planear compras de diciembre.
- **Electrificación:** 4 estados concentran el 56% de los híbridos/eléctricos; ahí conviene poner puntos de carga o agencias especializadas antes que la competencia.

*(escriban aquí la propuesta del equipo)*

## 2. Análisis de datos y metadatos (5%)
Explicar qué datos tenemos, de dónde salen y cómo están organizados. Una tabla como la del README del repo (archivo, página del PDF, filas, columnas, unidades, periodo) más lo que salga de `.info()` y `.describe()`.
Mencionen las dos cosas que ya sabemos de calidad de datos: el error de dedo de diciembre 2022 en el PDF y los huecos de `segmentos_agosto_2012_2026.csv` (no se inventó ningún número).

**Pista:** `data.info()`, `data.describe()`, `data.dtypes`. La columna `fecha` llega como texto: `pd.to_datetime(...)`.

In [ ]:
# Escribe aquí tu código


## 3. EDA — Análisis exploratorio (10%)
Aquí van la mayoría de las gráficas. Mostrar cómo están organizados y distribuidos los datos. Sugerencia de 6 gráficas, cada una en su celda:

**3.1 La serie completa (2008-2026).** Una línea, fecha contra unidades. Marquen 2009 y abril-mayo 2020 con `ax.axvline(...)` o `ax.text(...)`.
*Pista:* `fig, ax = plt.subplots(figsize=(12,4))` y `ax.plot(data.fecha, data.unidades)`.

In [ ]:
# Escribe aquí tu código


**3.2 Distribución.** Un histograma de las unidades mensuales y un boxplot por mes (`sns.boxplot(x="mes_num", y="unidades", data=data)`). Esto responde literalmente "cómo están distribuidos los datos".

In [ ]:
# Escribe aquí tu código


**3.3 Comparar años.** Eje x = mes, una línea por año (3 o 4 años, por ejemplo 2019, 2020, 2025, 2026). El color es un **marcador visual de Bertin**: variable cualitativa (el año).
*Pista:* un `for` sobre una lista de años, filtrar `data[data.anio == a]`, graficar `mes_num` contra `unidades` con `label=a`, y `ax.legend()`.

In [ ]:
# Escribe aquí tu código


**3.4 Mapa de calor mes × año.** Es la forma más rápida de ver estacionalidad (diciembre) y años malos.
*Pista:* `tabla = data.pivot_table(index="mes_num", columns="anio", values="unidades")` y `sns.heatmap(tabla, cmap="Greens")`. Aquí sí va la barra de color porque la variable es continua.

In [ ]:
# Escribe aquí tu código


**3.5 Segmentos, enero-agosto 2025 vs 2026.** Barras horizontales agrupadas con `segmentos_comparativo_2025_2026.csv` filtrando `tipo == "acumulado"`. El dato: usos múltiples (SUV) ya es 41.8% del mercado y compactos cae 5.7%.
*Pista:* `pivot_table(index="segmento", columns="periodo", values="unidades")` y `.plot.barh(ax=ax)`.

In [ ]:
# Escribe aquí tu código


**3.6 Top 10 marcas, importado vs nacional** con `marcas_origen_ene_ago_2025_2026.csv` (barras apiladas, color = origen). Y si les da tiempo, **híbridos y eléctricos 2016-2026** con `hibridos_electricos_2016_2026.csv` (barras de unidades + línea del %).
*Pista:* filtrar `anio == 2026`, `pivot_table(index="marca", columns="origen", values="unidades")`, total por fila, `.sort_values(...)`, `.head(10)`, `.plot.barh(stacked=True, ax=ax)`.

In [ ]:
# Escribe aquí tu código


## 4. Técnica de ciencia de datos (10%)
Edgar pide al menos una: aprendizaje supervisado, no supervisado, NLP o imágenes. Lo que ya vimos en clase es **supervisado**, así que esa es la apuesta segura.

**Opción A (recomendada) — Regresión para pronosticar ventas mensuales.**
Variable objetivo: `unidades`. Descriptores: `anio` (tendencia) y `mes_num` (estacionalidad). El mes es cualitativo, así que se convierte en columnas 0/1 con `pd.get_dummies(data["mes_num"], prefix="mes")`. Entrenar con 2010-2025 (o quitando 2020 y explicando por qué) y pronosticar septiembre-diciembre 2026.
Luego comparar contra el pronóstico oficial de AMDA (`pronostico_amda_2026.csv`, columna `estimado_2026_oficial`) y contra lo que AMDA estimó para enero-agosto vs lo que realmente pasó. Esa comparación es la "aportación".

*Pista:* `from sklearn.linear_model import LinearRegression`, `modelo.fit(X, y)`, `modelo.predict(X_nuevo)`. Para medir el error: `from sklearn.metrics import mean_absolute_error`.

**Opción B (extra) — Agrupar marcas (no supervisado).** Con `marcas_origen_ene_ago_2025_2026.csv` calcular por marca: unidades 2026, % importado y crecimiento 2025→2026, y correr `KMeans(n_clusters=3)`. Salen grupos tipo "chinas importadoras en crecimiento", "grandes con planta en México", etc.

In [ ]:
# 4.1 Preparar X (año + meses como 0/1) y y (unidades)


In [ ]:
# 4.2 Entrenar el modelo y revisar el error


In [ ]:
# 4.3 Pronosticar sep-dic 2026 y comparar con AMDA


## 5. Data storytelling y visualización (5%)
No es una gráfica nueva, es cómo se cuentan las de arriba. Tres reglas que Edgar valora:
1. El título de cada gráfica dice el hallazgo, no la variable ("Los SUV ya son 4 de cada 10 autos vendidos", no "Ventas por segmento").
2. Mismos colores para lo mismo en todas las gráficas (un color para 2025, otro para 2026; un color por segmento).
3. Un hilo: **qué pasaba → qué cambió → qué proponemos**. Escriban ese hilo aquí en 5 o 6 renglones y úsenlo para la presentación.

## 6. Conclusiones y aportaciones (5%)
Cuatro o cinco oraciones:
- **Lo que confirma la hipótesis** (y lo que no).
- **Lo que solo aplica a 2026** vs lo que es tendencia de años.
- **Lo que NO se puede saber con estos datos** (precios, márgenes, por qué cae compactos, inventario por agencia).
- **La aportación concreta** para el negocio que escogieron (una decisión que tomarían con esto).

## 7. Dashboard en Streamlit (10%, se entrega la liga)
Edgar valora que sea dinámico, atractivo y con **varios widgets**. El repo ya trae `main.py` que corre (título, slider de año y tabla). Ideas de widgets, cada uno mueve una gráfica de la sección 3:
- `st.slider("Año", 2008, 2026)` → gráfica 3.3 (ese año contra el anterior)
- `st.selectbox("Segmento", [...])` → serie de agosto de ese segmento
- `st.multiselect("Marcas", [...])` → gráfica 3.6 solo con las marcas elegidas
- `st.radio("Origen", ["Ambos", "Importado", "Nacional"])`
- `st.checkbox("Mostrar pronóstico AMDA")` → sobrepone el pronóstico a la serie
- `st.metric(...)` en `st.columns(3)` para los KPIs (ventas 2026, variación anual, % híbridos)
- `st.tabs([...])` para separar EDA / modelo / conclusiones

Se despliega igual que la app de pases: share.streamlit.io → Deploy → repo → `main.py`.

## 8. Presentación (15 minutos ± 2)
Fuera del rango es −10%, así que hay que cronometrarla. Guion sugerido:

| min | qué | quién |
|---|---|---|
| 0-2 | Propuesta de negocio y para quién | |
| 2-4 | Datos y metadatos (de dónde salen, qué traen, qué problemas tienen) | |
| 4-8 | EDA: 3 o 4 gráficas, no las 6 | |
| 8-11 | Modelo: qué predice, qué tan bien, comparación con AMDA | |
| 11-13 | Demo del dashboard en vivo (abrirlo antes, que ya esté cargado) | |
| 13-15 | Conclusiones y aportación | |